**Introduction**
In this assignment, I will use the traffic collision data from the City of Toronto to explore patterns in fatal vs non-fatal collisions.

In [1]:
from pathlib import Path
import pandas as pd
import requests

df = pd.read_csv("https://ckan0.cf.opendata.inter.prod-toronto.ca/dataset/0b6d3a00-7de1-440b-b47c-7252fd13929f/resource/fdb2834f-3a92-41dd-b098-fbd84acb9cfe/download/Motor%20Vehicle%20Collisions%20with%20KSI%20Data%20-%204326.csv")

df.columns

Index(['_id', 'ACCNUM', 'DATE', 'TIME', 'STREET1', 'STREET2', 'OFFSET',
       'ROAD_CLASS', 'DISTRICT', 'ACCLOC', 'TRAFFCTL', 'VISIBILITY', 'LIGHT',
       'RDSFCOND', 'ACCLASS', 'IMPACTYPE', 'INVTYPE', 'INVAGE', 'INJURY',
       'FATAL_NO', 'INITDIR', 'VEHTYPE', 'MANOEUVER', 'DRIVACT', 'DRIVCOND',
       'PEDTYPE', 'PEDACT', 'PEDCOND', 'CYCLISTYPE', 'CYCACT', 'CYCCOND',
       'PEDESTRIAN', 'CYCLIST', 'AUTOMOBILE', 'MOTORCYCLE', 'TRUCK',
       'TRSN_CITY_VEH', 'EMERG_VEH', 'PASSENGER', 'SPEEDING', 'AG_DRIV',
       'REDLIGHT', 'ALCOHOL', 'DISABILITY', 'HOOD_158', 'NEIGHBOURHOOD_158',
       'HOOD_140', 'NEIGHBOURHOOD_140', 'DIVISION', 'geometry'],
      dtype='object')

In [2]:
# Parse DATE and TIME
df['DATE'] = pd.to_datetime(df['DATE'], errors='coerce')

# Convert TIME to hour and ensure it's in the correct format
df['HOUR'] = df['TIME'].astype(str).str.zfill(4).str[:2].astype(int)

# Keep only meaningful accident classes
df = df[df['ACCLASS'].isin(['Fatal', 'Non-Fatal Injury'])]

In [3]:
hourly_severity = (
    df.groupby(['HOUR', 'ACCLASS']) # Group by HOUR and ACCLASS
    .size()     # Count the number of occurrences in each group
    .reset_index(name='COUNT')  # Reset index and name the count column 'COUNT'
)

In [7]:
import plotly.express as px

fig = px.line(
    hourly_severity,    # Use the aggregated data
    x='HOUR',           # X-axis is the hour of the day
    y='COUNT',          # Y-axis is the count of collisions
    color='ACCLASS',    # Different lines for each accident severity
    markers=True,       # Show markers on the lines
    title='Collisions by Severity and Hour of Day',     # Title of the plot
    labels={                # Axis labels
        'HOUR': 'Hour of Day',
        'COUNT': 'Number of Collisions',
        'ACCLASS': 'Accident Severity'
    }
)

fig.update_layout(          
    xaxis=dict(tickmode='linear',   # Show every hour on the x-axis
               range=[-0.5,23.5]    # Set x-axis range to show all hours from 0 to 23. -0.5 and 23.5 to ensure markers at the edges are visible
               ),
    hovermode='x unified'           # Show all hover info for a given x-value together
)

fig.show()